# Cross-Domain Scenario Preparation

This notebook prepares **two cross-domain YOLO training scenarios** for underwater fish detection:

1. **Scenario 1 – Degradation shift**  
   - Train on: **all domains except DeepFish**  
   - Test on: **DeepFish** (DeepFish positives + DeepFish negatives treated as a single domain)

2. **Scenario 2 – Colour / habitat shift**  
   - Train on: **all domains except luderick**  
   - Test on: **luderick**

For both scenarios, the notebook:
- Creates an 80/20 **train/validation** split.
- Uses a **two-step stratified procedure**:
  1. Split **per dataset** to preserve dataset representation.
  2. Stratify within each dataset by **positive vs negative** images (`has_fish`).
- Treats **DeepFish** as a single domain combining `deepfish` and `deepfish_negatives`.
- Writes out scenario-specific YOLO folder structures and `data.yaml` files.

## 1. Imports and global configuration

In [2]:
import os
from pathlib import Path
from typing import Dict, List, Tuple

import shutil
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

## 2. Define paths and dataset configuration

Adjust `ROOT` and `DATASETS_ROOT` if needed. Each dataset is assumed to be stored in a
YOLO-like structure:

```text
<DATASETS_ROOT>/<dataset_name>/images
<DATASETS_ROOT>/<dataset_name>/labels
```

DeepFish is represented by two folders (`deepfish`, `deepfish_negatives`) but treated as one domain.

In [3]:
# Project root (defaults to the current working directory).
# You can override with environment variables if your data lives elsewhere.
ROOT = Path(os.getenv("HT_VISION_ROOT", Path.cwd())).resolve()

# Where the original per-dataset YOLO-style datasets live
DATASETS_ROOT = Path(
    os.getenv("HT_VISION_DATASETS_ROOT", str(ROOT / "data" / "OriginalCleaned_DS"))
).resolve()

# Output root for the prepared scenarios
SCENARIOS_ROOT = Path(
    os.getenv("HT_VISION_SCENARIOS_ROOT", str(ROOT / "cross_domain_scenarios_datasets"))
).resolve()
SCENARIOS_ROOT.mkdir(parents=True, exist_ok=True)

# Canonical dataset list
DATASET_NAMES = [
    "AquaCoop",
    "luderick",
    "OzFish",
    "aquarium",
    "deepfish",
    "deepfish_negatives",
    "fishclef",
    "fish_416",
    "f4k",
]

# Map each dataset to its image/label directories
DATASETS: Dict[str, Dict[str, Path]] = {
    name: {
        "images": DATASETS_ROOT / name / "images",
        "labels": DATASETS_ROOT / name / "labels",
    }
    for name in DATASET_NAMES
}

# Domain mapping (DeepFish domain = deepfish + deepfish_negatives)
DOMAIN_MAP: Dict[str, str] = {
    "AquaCoop": "AquaCoop",
    "luderick": "luderick",
    "OzFish": "OzFish",
    "aquarium": "aquarium",
    "deepfish": "DeepFish",
    "deepfish_negatives": "DeepFish",
    "fishclef": "fishclef",
    "fish_416": "fish_416",
    "f4k": "f4k",
}

# Single-class setup (YOLO)
CLASS_NAMES = {0: "fish"}


## 3. Label utilities and fish presence detection

We ensure that every image has a corresponding label `.txt` file. Missing labels are
created as empty files, representing **negative images** (no fish).

In [4]:
def ensure_label_file(label_path: Path) -> None:
    """Ensure a YOLO label file exists.
    If missing, create an empty file (negative image: no fish).
    """
    if not label_path.exists():
        label_path.parent.mkdir(parents=True, exist_ok=True)
        label_path.touch()


def has_fish_in_label(label_path: Path) -> bool:
    """Return True if the YOLO label file contains at least one annotation line.
    Empty file or non-existing file -> no fish.
    """
    if not label_path.exists():
        return False
    try:
        text = label_path.read_text().strip()
    except UnicodeDecodeError:
        # If something weird happens, assume no fish (conservative)
        return False
    return len(text) > 0

## 4. Build a master index of all images

We build a DataFrame with one row per image across all datasets, including:
- `dataset`
- `domain` (merging DeepFish positive/negative datasets)
- `image_path`
- `label_path`
- `has_fish` (boolean)

This index is the basis for scenario-specific splits.

In [5]:
def build_master_index(datasets_cfg: Dict[str, Dict[str, Path]]) -> pd.DataFrame:
    rows = []
    for ds_name, paths in datasets_cfg.items():
        img_dir = paths["images"]
        lbl_dir = paths["labels"]

        if not img_dir.exists():
            print(f"[WARN] Images dir does not exist for {ds_name}: {img_dir}")
            continue

        image_files = sorted(
            [p for p in img_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
        )

        print(f"{ds_name}: found {len(image_files)} images")

        for img_path in image_files:
            # label txt with same stem, under labels directory mirroring relative path
            rel = img_path.relative_to(img_dir)  # e.g. sub/dir/img001.jpg
            label_path = lbl_dir / rel.with_suffix(".txt")

            # Ensure label file exists (create empty if needed)
            ensure_label_file(label_path)

            rows.append(
                {
                    "dataset": ds_name,
                    "domain": DOMAIN_MAP[ds_name],
                    "image_path": img_path,
                    "label_path": label_path,
                    "has_fish": has_fish_in_label(label_path),
                }
            )

    df = pd.DataFrame(rows)
    print("Total images in master index:", len(df))
    return df


master_index = build_master_index(DATASETS)
master_index.head()

AquaCoop: found 1238 images
luderick: found 4276 images
OzFish: found 350 images
aquarium: found 637 images
deepfish: found 4505 images
deepfish_negatives: found 2012 images
deepfish: found 4505 images
deepfish_negatives: found 2012 images
fishclef: found 14273 images
fishclef: found 14273 images
fish_416: found 680 images
f4k: found 794 images
Total images in master index: 28765
fish_416: found 680 images
f4k: found 794 images
Total images in master index: 28765


,dataset,domain,image_path,label_path,has_fish
0,AquaCoop,AquaCoop,/mnt/Data1/mpiccolo/HT_Vision/HT_Vision_Datase...,/mnt/Data1/mpiccolo/HT_Vision/HT_Vision_Datase...,True
1,AquaCoop,AquaCoop,/mnt/Data1/mpiccolo/HT_Vision/HT_Vision_Datase...,/mnt/Data1/mpiccolo/HT_Vision/HT_Vision_Datase...,True
2,AquaCoop,AquaCoop,/mnt/Data1/mpiccolo/HT_Vision/HT_Vision_Datase...,/mnt/Data1/mpiccolo/HT_Vision/HT_Vision_Datase...,True
3,AquaCoop,AquaCoop,/mnt/Data1/mpiccolo/HT_Vision/HT_Vision_Datase...,/mnt/Data1/mpiccolo/HT_Vision/HT_Vision_Datase...,True
4,AquaCoop,AquaCoop,/mnt/Data1/mpiccolo/HT_Vision/HT_Vision_Datase...,/mnt/Data1/mpiccolo/HT_Vision/HT_Vision_Datase...,True


## 5. Define training and test domains for each scenario

- **Scenario 1**: train on all domains except DeepFish, test on DeepFish.
- **Scenario 2**: train on all domains except luderick, test on luderick.

We derive the training domains automatically from the full domain list.

In [6]:
# Scenario definitions on domain level
SCENARIOS = {
    "scenario1_all_except_deepfish_test_deepfish": {
        "test_domains": ["DeepFish"],  # deepfish + deepfish_negatives
    },
    "scenario2_all_except_luderick_test_luderick": {
        "test_domains": ["luderick"],
    },
}

# Derive train domains automatically for each scenario
all_domains = sorted(master_index["domain"].unique().tolist())
print("All domains:", all_domains)

for scen_name, cfg in SCENARIOS.items():
    test_domains = cfg["test_domains"]
    train_domains = [d for d in all_domains if d not in test_domains]
    cfg["train_domains"] = train_domains
    print(f"{scen_name} -> train: {train_domains}, test: {test_domains}")

All domains: ['AquaCoop', 'DeepFish', 'OzFish', 'aquarium', 'f4k', 'fish_416', 'fishclef', 'luderick']
scenario1_all_except_deepfish_test_deepfish -> train: ['AquaCoop', 'OzFish', 'aquarium', 'f4k', 'fish_416', 'fishclef', 'luderick'], test: ['DeepFish']
scenario2_all_except_luderick_test_luderick -> train: ['AquaCoop', 'DeepFish', 'OzFish', 'aquarium', 'f4k', 'fish_416', 'fishclef'], test: ['luderick']


## 6. Two-step stratified 80/20 train/validation split

We implement a **two-step** stratified split:
1. Split **separately for each dataset** to preserve dataset representation in train/val.
2. Within each dataset, use `StratifiedShuffleSplit` on `has_fish` to balance positives/negatives.

If a dataset is too small or has only one class, we fall back to a simple random split.

In [7]:
def stratified_train_val_split_per_dataset(
    df: pd.DataFrame,
    train_frac: float = 0.8,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Perform a two-step stratified split:
      1. Split separately per dataset (ensures balanced representation across datasets).
      2. Within each dataset, use StratifiedShuffleSplit on has_fish (pos/neg balance).
    Returns global train_df, val_df concatenated over all datasets.
    """
    train_parts = []
    val_parts = []

    for ds_name, df_ds in df.groupby("dataset"):
        y = df_ds["has_fish"].astype(int).values

        if len(df_ds) < 3 or len(np.unique(y)) == 1:
            # fallback: simple (non-stratified) split for tiny or single-class group
            n_train = int(round(len(df_ds) * train_frac))
            df_ds_shuffled = df_ds.sample(frac=1.0, random_state=random_state)
            train_parts.append(df_ds_shuffled.iloc[:n_train])
            val_parts.append(df_ds_shuffled.iloc[n_train:])
            print(f"[WARN] Non-stratified split for dataset {ds_name} (too few or single class).")
            continue

        splitter = StratifiedShuffleSplit(
            n_splits=1, train_size=train_frac, random_state=random_state
        )
        idx = np.arange(len(df_ds))
        train_idx, val_idx = next(splitter.split(idx, y))

        train_parts.append(df_ds.iloc[train_idx])
        val_parts.append(df_ds.iloc[val_idx])

        print(
            f"{ds_name}: total={len(df_ds)}, train={len(train_idx)}, val={len(val_idx)}, "
            f"pos_train={df_ds.iloc[train_idx]['has_fish'].sum()}, "
            f"pos_val={df_ds.iloc[val_idx]['has_fish'].sum()}"
        )

    train_df = pd.concat(train_parts, ignore_index=True)
    val_df = pd.concat(val_parts, ignore_index=True)

    print("Global train size:", len(train_df))
    print("Global val size:", len(val_df))
    return train_df, val_df

## 7. File copy helpers and `data.yaml` writer

These utilities copy images/labels into scenario-specific folders and generate a
standard YOLO `data.yaml` file.

In [8]:
def copy_image_and_label(row: pd.Series, dest_img_dir: Path, dest_lbl_dir: Path) -> None:
    """Copy image and its label to destination folders, preserving filename.
    If you want to preserve subfolder structure, adapt this accordingly.
    """
    src_img = Path(row["image_path"])
    src_lbl = Path(row["label_path"])

    dest_img_dir.mkdir(parents=True, exist_ok=True)
    dest_lbl_dir.mkdir(parents=True, exist_ok=True)

    dst_img = dest_img_dir / src_img.name
    dst_lbl = dest_lbl_dir / src_lbl.name

    shutil.copy2(src_img, dst_img)
    shutil.copy2(src_lbl, dst_lbl)


def write_data_yaml(
    root_dir: Path,
    yaml_path: Path,
    class_names: Dict[int, str],
    train_rel: str = "images/train",
    val_rel: str = "images/val",
    test_rel: str = "images/test",
) -> None:
    """Write a YOLO data.yaml file pointing to train/val/test and class names."""
    lines = []
    lines.append(f"path: {root_dir.as_posix()}")
    lines.append(f"train: {train_rel}")
    lines.append(f"val: {val_rel}")
    lines.append(f"test: {test_rel}")
    lines.append("names:")
    for k, v in class_names.items():
        lines.append(f"  {k}: {v}")
    yaml_text = "\n".join(lines) + "\n"

    yaml_path.write_text(yaml_text)
    print(f"Written data.yaml -> {yaml_path}")

## 8. Build both scenarios

For each scenario we will:
1. Select the **train pool** (all images from training domains).
2. Select the **test set** (all images from test domains).
3. Perform a two-step stratified 80/20 train/val split on the train pool.
4. Copy images/labels into a scenario-specific YOLO structure:

```text
cross_domain_scenarios/
  <scenario_name>/
    images/train, images/val, images/test
    labels/train, labels/val, labels/test
    data.yaml
```

In [ ]:
for scen_name, cfg in SCENARIOS.items():
    print("\n" + "=" * 80)
    print("Preparing scenario:", scen_name)
    print("=" * 80)

    scen_root = SCENARIOS_ROOT / scen_name
    # Rebuild the scenario folder from scratch
    if scen_root.exists():
        print(f"[INFO] Removing existing scenario folder: {scen_root}")
        shutil.rmtree(scen_root)
    scen_root.mkdir(parents=True, exist_ok=True)

    # 1) Filter master_index into train-pool and test-set based on domains
    train_domains = cfg["train_domains"]
    test_domains = cfg["test_domains"]

    train_pool = master_index[master_index["domain"].isin(train_domains)].copy()
    test_set = master_index[master_index["domain"].isin(test_domains)].copy()

    print(f"Train pool size (all train domains): {len(train_pool)}")
    print(f"Test set size (all test domains): {len(test_set)}")

    # 2) Stratified train/val split on train_pool (per-dataset, stratified by has_fish)
    train_df, val_df = stratified_train_val_split_per_dataset(train_pool)

    # 3) Prepare destination dirs
    img_train_dir = scen_root / "images" / "train"
    img_val_dir = scen_root / "images" / "val"
    img_test_dir = scen_root / "images" / "test"

    lbl_train_dir = scen_root / "labels" / "train"
    lbl_val_dir = scen_root / "labels" / "val"
    lbl_test_dir = scen_root / "labels" / "test"

    # 4) Copy train images/labels
    print("Copying TRAIN files...")
    for _, row in train_df.iterrows():
        copy_image_and_label(row, img_train_dir, lbl_train_dir)

    # 5) Copy val images/labels
    print("Copying VAL files...")
    for _, row in val_df.iterrows():
        copy_image_and_label(row, img_val_dir, lbl_val_dir)

    # 6) Copy test images/labels (entire test domain, no splitting)
    print("Copying TEST files (full test domain)...")
    for _, row in test_set.iterrows():
        copy_image_and_label(row, img_test_dir, lbl_test_dir)

    # 7) Write data.yaml for this scenario
    yaml_path = scen_root / "data.yaml"
    write_data_yaml(
        root_dir=scen_root,
        yaml_path=yaml_path,
        class_names=CLASS_NAMES,
        train_rel="images/train",
        val_rel="images/val",
        test_rel="images/test",
    )

    print(f"[DONE] Scenario prepared at: {scen_root}")



Preparing scenario: scenario1_all_except_deepfish_test_deepfish
Train pool size (all train domains): 22248
Test set size (all test domains): 6517
[WARN] Non-stratified split for dataset AquaCoop (too few or single class).
[WARN] Non-stratified split for dataset OzFish (too few or single class).
aquarium: total=637, train=509, val=128, pos_train=300, pos_val=76
[WARN] Non-stratified split for dataset f4k (too few or single class).
[WARN] Non-stratified split for dataset fish_416 (too few or single class).
[WARN] Non-stratified split for dataset fishclef (too few or single class).
[WARN] Non-stratified split for dataset luderick (too few or single class).
Global train size: 17797
Global val size: 4451
Copying TRAIN files...
Copying VAL files...
Copying VAL files...
Copying TEST files (full test domain)...
Copying TEST files (full test domain)...
Written data.yaml -> /mnt/Data1/mpiccolo/HT_Vision/Controlled_cross_domain_generalization/cross_domain_scenarios_datasets/scenario1_all_except_

## 9. Sanity checks

Quick checks for image/label counts in each split and scenario.

In [12]:
for scen_name in SCENARIOS.keys():
    scen_root = SCENARIOS_ROOT / scen_name
    print("\n=== START SCENARIO:", scen_name, "===")
    for split in ["train", "val", "test"]:
        img_dir = scen_root / "images" / split
        lbl_dir = scen_root / "labels" / split
        n_img = (
            len(list(img_dir.glob("*.jpg")))
            + len(list(img_dir.glob("*.jpeg")))
            + len(list(img_dir.glob("*.png")))
        )
        n_lbl = len(list(lbl_dir.glob("*.txt")))
        print(f"  {split}: images={n_img}, labels={n_lbl}")
    print("=== END SCENARIO:", scen_name, "===\n")



=== START SCENARIO: scenario1_all_except_deepfish_test_deepfish ===
  train: images=17797, labels=17797
  val: images=4451, labels=4451
  test: images=6517, labels=6517
=== END SCENARIO: scenario1_all_except_deepfish_test_deepfish ===


=== START SCENARIO: scenario2_all_except_luderick_test_luderick ===
  train: images=19590, labels=19590
  val: images=4899, labels=4899
  test: images=4276, labels=4276
=== END SCENARIO: scenario2_all_except_luderick_test_luderick ===

  train: images=19590, labels=19590
  val: images=4899, labels=4899
  test: images=4276, labels=4276
=== END SCENARIO: scenario2_all_except_luderick_test_luderick ===

